# `RunnableBinding: RunnableBindingBase[Input, Output]`

`RunnableBinding` wraps another `Runnable` and adds fixed arguments, configuration, listeners, custom types, or retry behaviour without removing batching, streaming, or asynchronous support.

It is normally created through methods such as `bind()`, `with_config()`, `with_listeners()`, `with_types()`, or `with_retry()`.

## Type Parameters

```python
Input # Input type accepted by the wrapped Runnable
Output # Output type produced by the wrapped Runnable
```

## Fields

```python
bound: Runnable[Input, Output] # Underlying Runnable receiving delegated operations
kwargs: Mapping[str, Any] # Keyword arguments supplied automatically during execution
config: RunnableConfig # Configuration merged into call-time configuration
config_factories: list[Callable[[RunnableConfig], RunnableConfig]] # Functions that generate additional configuration
custom_input_type: Any | None # Optional replacement input type
custom_output_type: Any | None # Optional replacement output type
```

## Constructor

Direct construction is supported, although bindings are normally created through Runnable helper methods.

```python
RunnableBinding(
    *,
    bound: Runnable[Input, Output], # Underlying Runnable receiving delegated calls
    kwargs: Mapping[str, Any] | None = None, # Keyword arguments permanently bound to the Runnable
    config: RunnableConfig | None = None, # Configuration permanently bound to the Runnable
    config_factories: list[Callable[[RunnableConfig], RunnableConfig]] | None = None, # Functions that derive additional configuration
    custom_input_type: type[Input] | BaseModel | None = None, # Optional replacement input type
    custom_output_type: type[Output] | BaseModel | None = None, # Optional replacement output type
    **other_kwargs: Any, # Additional serializable model fields
) -> None # Initialize the RunnableBinding
```

## Overridden Methods

### `bind`

Returns a new binding containing additional fixed keyword arguments.

Newly supplied values replace existing bound values with the same key.

### `with_config`

Returns a new binding containing additional runtime configuration.

The supplied configuration is merged with configuration already stored by the binding.

### `with_listeners`

Returns a new binding containing synchronous lifecycle listeners.

Listeners may run when execution starts, completes successfully, or raises an error.

### `with_types`

Returns a new binding with custom input or output types.

An omitted type preserves the corresponding custom type already stored by the binding.

### `with_retry`

Returns a new binding whose underlying Runnable has retry behaviour applied.

The existing bound arguments and configuration are preserved.

## Attribute Delegation

Attributes not defined by `RunnableBinding` are obtained from the underlying `Runnable`.

When a delegated callable accepts a `config` parameter, the binding merges its stored configuration into that parameter automatically.

## Inherited Execution Behaviour

### `get_name`

Returns the name of the underlying `Runnable`.

### `InputType`

Returns the custom input type when configured; otherwise, returns the underlying Runnable input type.

### `OutputType`

Returns the custom output type when configured; otherwise, returns the underlying Runnable output type.

### `get_input_schema`

Returns a schema based on the custom input type or delegates schema generation to the underlying Runnable.

### `get_output_schema`

Returns a schema based on the custom output type or delegates schema generation to the underlying Runnable.

### `config_specs`

Returns the configurable-field specifications exposed by the underlying Runnable.

### `get_graph`

Returns the graph of the underlying Runnable using merged configuration.

### `invoke`

Synchronously invokes the underlying Runnable with merged configuration and keyword arguments.

Call-time keyword arguments override bound keyword arguments.

### `ainvoke`

Asynchronously invokes the underlying Runnable with merged configuration and keyword arguments.

Call-time keyword arguments override bound keyword arguments.

### `batch`

Delegates synchronous batch execution while applying the binding to every input.

### `abatch`

Delegates asynchronous batch execution while applying the binding to every input.

### `batch_as_completed`

Yields indexed synchronous batch results as individual inputs finish.

### `abatch_as_completed`

Asynchronously yields indexed batch results as individual inputs finish.

### `stream`

Synchronously streams output from the underlying Runnable with bound arguments and configuration.

### `astream`

Asynchronously streams output from the underlying Runnable with bound arguments and configuration.

### `stream_events`

Forwards synchronous event streaming while preserving bound arguments and configuration.

### `astream_events`

Forwards asynchronous event streaming while preserving bound arguments and configuration.

### `transform`

Transforms a synchronous input iterator through the underlying Runnable.

### `atransform`

Transforms an asynchronous input iterator through the underlying Runnable.

## Behaviour

- The original Runnable is not modified.
- Every modifier returns a new binding.
- Call-time keyword arguments override stored keyword arguments.
- Stored configuration is merged with call-time configuration.
- The wrapped Runnable retains its synchronous, asynchronous, batch, and streaming capabilities.

In [ ]:
from langchain_core.runnables import RunnableBinding, RunnableLambda # Import the required Runnable classes

def add_prefix(text: str, prefix: str) -> str: # Define a function that accepts input text and a prefix
    return prefix + text # Return the prefix combined with the input text

base_runnable = RunnableLambda(add_prefix) # Convert the Python function into a Runnable

binding = RunnableBinding( # Create a RunnableBinding directly
    bound=base_runnable, # Set the underlying Runnable
    kwargs={"prefix": "Result: "}, # Permanently bind the prefix argument
) # Finish creating the RunnableBinding

result = binding.invoke("Hello LangChain") # Execute without passing the prefix again

print(result) # Display the final result